# Compact Analysis

Runs the compact benchmark comparison, then validates the saved compact strategies under actual Helstrom branch probabilities.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# Resolve paths whether the notebook is launched from the repository root or this folder.
working_directory = Path.cwd().resolve()
PROJECT_DIRECTORY = (
    working_directory / "min-error-sequential"
    if (working_directory / "min-error-sequential").is_dir()
    else working_directory
)
MODULE_DIRECTORY = PROJECT_DIRECTORY
SOURCE_DIRECTORY = PROJECT_DIRECTORY / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

# Reload algorithm code first so comparison_analysis sees the latest heuristic signature.
import heuristic
heuristic = importlib.reload(heuristic)

# Reload local analysis code when this notebook is already open.
import comparison_analysis
comparison_analysis = importlib.reload(comparison_analysis)
field_family_summary = comparison_analysis.field_family_summary
load_paired_results = comparison_analysis.load_paired_results
overall_error_curve_summary = comparison_analysis.overall_error_curve_summary
paired_summary = comparison_analysis.paired_summary
runtime_by_k_summary = comparison_analysis.runtime_by_k_summary
run_paired_dataset = comparison_analysis.run_paired_dataset

print("Project directory:", PROJECT_DIRECTORY)
print("Source directory:", SOURCE_DIRECTORY)


## Run Settings

The `RUN_SCOPE` setting selects either the balanced pilot run (`"test"`) or all compact records (`"full"`). The test scope uses 10 replicates from each field-family, $K$, and $M$ cell, giving 180 cases per dataset. The full scope uses all 2,520 cases per dimension and is much slower because every case also runs Exhaustive search.


In [ ]:
# Run both spatial benchmark datasets. Each dataset contains 2,520 cases.
DATASETS = ["compact_benchmark_1d", "compact_benchmark_2d"]

# Switch here: "test" runs 180 balanced cases; "full" runs all 2,520 cases per dataset.
RUN_SCOPE = "test"
if RUN_SCOPE not in {"test", "full"}:
    raise ValueError('RUN_SCOPE must be "test" or "full"')

# None keeps every replicate; 10 gives the balanced test subset.
REPLICATES_PER_CELL = None if RUN_SCOPE == "full" else 10

PROGRESS_EVERY = 1 if RUN_SCOPE == "full" else 1

# Save result tables and figures under this directory.
RESULTS_ROOT = MODULE_DIRECTORY / "results" / "cmp_compact"

print("Run scope:", RUN_SCOPE)
print("Cases per dataset:", "2,520" if RUN_SCOPE == "full" else "180")
print("Progress interval:", PROGRESS_EVERY, "cases")
print("Results directory:", RESULTS_ROOT)


## Algorithm Configuration



In [ ]:
# Exhaustive search has no optional tuning parameters.
OPTIMAL_PARAMETERS = {}

# The existing greedy method has no optional tuning parameters.
GREEDY_PARAMETERS = {}

# Configure allocation VND and final beam-tree construction.
HEURISTIC_PARAMETERS = {
    "beam_width": 3,  # Number of partial trees kept after each expansion.
    "initialization": "near_uniform",  # Start VND from a near-uniform allocation.
    "move_sizes": (1, 2, 3),  # VND neighborhood levels: transfer L sensors.
}

# Use the same seed as the heuristic's near-uniform starting allocation.
UNIFORM_PARAMETERS = {
    "random_seed": 0,
}

# Reproduce case-specific random allocations and decision trees.
RANDOM_PARAMETERS = {
    "random_seed": 0,
}

print("Optimal parameters:", OPTIMAL_PARAMETERS)
print("Greedy parameters:", GREEDY_PARAMETERS)
print("Heuristic parameters:", HEURISTIC_PARAMETERS)
print("Uniform parameters:", UNIFORM_PARAMETERS)


## Algorithm Evaluation



In [ ]:
# Keep each dimension separate so the plots can show 1D and 2D side by side.
records_by_dataset = {}
for dataset in DATASETS:
    display(Markdown(f"### Running `{dataset}`"))
    # Run all algorithms using the same cases and budgets.
    records_by_dataset[dataset] = run_paired_dataset(
        dataset,  # Name of the input benchmark directory.
        replicates_per_cell=REPLICATES_PER_CELL,  # None means all cases.
        results_root=RESULTS_ROOT,  # Base directory for comparison results.
        optimal_parameters=OPTIMAL_PARAMETERS,
        greedy_parameters=GREEDY_PARAMETERS,
        heuristic_parameters=HEURISTIC_PARAMETERS,
        uniform_parameters=UNIFORM_PARAMETERS,
        random_parameters=RANDOM_PARAMETERS,
        progress_every=PROGRESS_EVERY,
    )


## Paired Surrogate-Error Summary

This table compares each approximate method against the surrogate-exhaustive solution on paired runs. Error gaps are `method surrogate error - Exhaustive surrogate error`; smaller values are better. Exact-error matches count runs where the method reaches the same surrogate error as Exhaustive within the configured tolerance.


In [ ]:
# Start a Markdown table for the main quality and runtime statistics.
summary_lines = [
    "| Dataset | Method | Cases | Paired runs | Mean surrogate-error gap | Maximum surrogate-error gap | Exact surrogate-error matches | Median speedup |",
    "|---|---|---:|---:|---:|---:|---:|---:|",
]
for dataset, records in records_by_dataset.items():
    # Calculate statistics from paired runs within one spatial dimension.
    summary = paired_summary(records)
    # The prefix selects the matching values from the summary dictionary.
    for method, prefix in (
        ("Greedy", "greedy"),
        ("Heuristic", "heuristic"),
        ("Uniform", "uniform"),
        ("Random", "random"),
    ):
        summary_lines.append(
            f"| {dataset} | {method} | {summary['cases']} | {summary['runs']} | "
            f"{summary[f'{prefix}_mean_gap']:.4f} | "
            f"{summary[f'{prefix}_maximum_gap']:.4f} | "
            f"{100 * summary[f'{prefix}_exact_match_rate']:.1f}% | "
            f"{summary[f'{prefix}_median_speedup']:.1f}x |"
        )
# Render the completed table inside the notebook.

display(Markdown("\n".join(summary_lines)))


## Overall Surrogate-Error Curve Summary

Each row is one point in the overall mean-error curves. The error columns are surrogate final errors from the fixed decision trees. Gap cells show absolute error reductions and percentage reductions; positive values mean the second method is better.


In [ ]:
curve_summary_lines = [
    r"| Dimension | Budget level | Mean $N/N_{\min}^{opt}$ | Optimal mean surrogate error | Greedy mean surrogate error | Heuristic mean surrogate error | Uniform mean surrogate error | Random mean surrogate error | Optimal reduction from Greedy | Optimal reduction from Heuristic | Heuristic reduction from Greedy |",
    "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|",
]
for records in records_by_dataset.values():
    for row in overall_error_curve_summary(records):
        curve_summary_lines.append(
            f"| {row['dimension']} | {row['budget_level']} | "
            f"{row['mean_budget_fraction']:.3f} | "
            f"{row['optimal_mean_error']:.4f} | "
            f"{row['greedy_mean_error']:.4f} | "
            f"{row['heuristic_mean_error']:.4f} | "
            f"{row['uniform_mean_error']:.4f} | "
            f"{row['random_mean_error']:.4f} | "
            f"{row['optimal_from_greedy']:.4f} "
            f"({row['optimal_from_greedy_percent']:.1f}%) | "
            f"{row['optimal_from_heuristic']:.4f} "
            f"({row['optimal_from_heuristic_percent']:.1f}%) | "
            f"{row['heuristic_from_greedy']:.4f} "
            f"({row['heuristic_from_greedy_percent']:.1f}%) |"
        )

display(Markdown("\n".join(curve_summary_lines)))


## Runtime Summary by Number of Fields



In [ ]:
runtime_summary_lines = [
    "| Dimension | $K$ | Optimal median runtime (s) | Greedy runtime (speedup) | Heuristic runtime (speedup) | Uniform runtime (speedup) | Random runtime (speedup) |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for records in records_by_dataset.values():
    for row in runtime_by_k_summary(records):
        runtime_summary_lines.append(
            f"| {row['dimension']} | {row['K']} | "
            f"{row['optimal_median_seconds']:.6f} | "
            f"{row['greedy_median_seconds']:.6f} "
            f"({row['greedy_median_speedup']:.1f}x) | "
            f"{row['heuristic_median_seconds']:.6f} "
            f"({row['heuristic_median_speedup']:.1f}x) | "
            f"{row['uniform_median_seconds']:.6f} "
            f"({row['uniform_median_speedup']:.1f}x) | "
            f"{row['random_median_seconds']:.6f} "
            f"({row['random_median_speedup']:.1f}x) |"
        )

display(Markdown("\n".join(runtime_summary_lines)))


## Error-Gap Summary by Field Family



In [ ]:
family_summary_lines = [
    "| Dimension | Field family | Greedy mean surrogate-error gap | Heuristic mean surrogate-error gap | Uniform mean surrogate-error gap | Random mean surrogate-error gap | Heuristic gap reduction from Greedy |",
    "|---|---|---:|---:|---:|---:|---:|",
]
for records in records_by_dataset.values():
    for row in field_family_summary(records):
        family_summary_lines.append(
            f"| {row['dimension']} | {row['field_family']} | "
            f"{row['greedy_mean_gap']:.4f} | "
            f"{row['heuristic_mean_gap']:.4f} | "
            f"{row['uniform_mean_gap']:.4f} | "
            f"{row['random_mean_gap']:.4f} | "
            f"{row['heuristic_gap_reduction']:.4f} "
            f"({row['heuristic_gap_reduction_percent']:.1f}%) |"
        )

display(Markdown("\n".join(family_summary_lines)))


## Comparison Plots

These figures use surrogate final error, because this notebook compares algorithms under the objective used for optimization.

1. **Mean surrogate-error curves:** quality-budget tradeoff separately for each $K$.
2. **Overall mean surrogate-error curves:** 1D and 2D summaries averaged across all included $K$, $M$, and field families.
3. **Runtime speedup:** computational benefit relative to exhaustive search.
4. **Surrogate-error gap by field family:** Gaussian, sinusoidal, and quadratic fields.


In [ ]:
# Close old figures, then reload and apply the shared style.
plt.close("all")
import plot_style
plot_style = importlib.reload(plot_style)
plot_style.configure_plot_style()
# Reload plotting code so an open kernel does not reuse older functions.
comparison_analysis = importlib.reload(comparison_analysis)
# Load saved records when the expensive run cell was intentionally skipped.
if "records_by_dataset" not in globals():
    records_by_dataset = {
        dataset: comparison_analysis.load_paired_results(
            dataset, results_root=RESULTS_ROOT, run_scope=RUN_SCOPE
        )
        for dataset in DATASETS
    }
# Create the four selected figures and save PNG and PDF copies.
for title, figure in comparison_analysis.make_combined_comparison_plots(
    records_by_dataset,  # Paired results for both dimensions.
    results_root=RESULTS_ROOT,  # Base directory for comparison results.
    run_scope=RUN_SCOPE,  # Keeps test and full-run summaries separate.
):
    display(Markdown(f"## {title}"))
    display(figure)
    plt.close(figure)


## Saved Files

The notebook saves detailed tables separately by dimension and puts the side-by-side figures in one combined folder:

```text
results/cmp_compact/
├── compact_1d_<test|full>/tables/
├── compact_2d_<test|full>/tables/
└── summary_compact_1d_compact_2d_<test|full>/figures/
    ├── mean_error_curves.*
    ├── runtime_speedup.*
    └── error_gap_by_field_family.*
```



## Actual-Error Validation

This section uses the compact comparison results saved above and reevaluates the fixed allocations and trees using actual branch probabilities.


In [ ]:
import json
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# Resolve paths whether the notebook is launched from the repository root or this folder.
working_directory = Path.cwd().resolve()
project_directory = (
    working_directory / "min-error-sequential"
    if (working_directory / "min-error-sequential").is_dir()
    else working_directory
)
source_directory = project_directory / "src"
if str(source_directory) not in sys.path:
    sys.path.insert(0, str(source_directory))

# Reload algorithm modules before reloading the analysis wrapper.
import heuristic
heuristic = importlib.reload(heuristic)

import actual_error_analysis
actual_error_analysis = importlib.reload(actual_error_analysis)

run_actual_error = actual_error_analysis.run_actual_error_evaluation
actual_error_summary = actual_error_analysis.actual_error_summary
actual_method_comparison_summary = actual_error_analysis.actual_method_comparison_summary
surrogate_quality_summary = actual_error_analysis.surrogate_quality_summary
bias_mae_summary = actual_error_analysis.bias_mae_summary
within_budget_correlation_summary = actual_error_analysis.within_budget_correlation_summary
method_ranking_agreement_summary = actual_error_analysis.method_ranking_agreement_summary
residual_by_k_summary = actual_error_analysis.residual_by_k_summary

print("Project directory:", project_directory)
print("Source directory:", source_directory)
print("Using actual_error_analysis from:", actual_error_analysis.__file__)


## Settings

The `COMPACT_RUN_SCOPE` setting selects either saved compact pilot records (`"test"`) or saved full compact records (`"full"`). This notebook is compact-only and includes Exhaustive, Greedy, Allocation + tree heuristic, Uniform, and Random because those strategies are saved by the compact comparison pipeline.



In [ ]:
# Compact data: includes Exhaustive, Greedy, Allocation + tree heuristic, Uniform, and Random.
COMPACT_DATASETS = ("compact_benchmark_1d", "compact_benchmark_2d")
COMPACT_RUN_SCOPE = "test"  # switch to "test" for the compact pilot records
EXPECTED_HEURISTIC_BEAM_WIDTH = 3

if COMPACT_RUN_SCOPE not in {"test", "full"}:
    raise ValueError('COMPACT_RUN_SCOPE must be "test" or "full"')

COMPACT_RESULTS_ROOT = project_directory / "results" / "cmp_compact"
ACTUAL_RESULTS_ROOT = project_directory / "results" / "actual_error"

print("Compact scope:", COMPACT_RUN_SCOPE)
print("Expected heuristic beam width:", EXPECTED_HEURISTIC_BEAM_WIDTH)
print("Reading compact saved allocations/trees from:", COMPACT_RESULTS_ROOT)
print("Writing analysis outputs under:", ACTUAL_RESULTS_ROOT)


## Compute Compact Actual Errors

This section loads the saved compact allocations and decision trees from the compact comparison run. It then reevaluates each fixed tree with actual Helstrom branch probabilities. The progress messages count raw method-run records, not newly generated problem instances.

In [ ]:
def validate_heuristic_beam_width(dataset, run_scope, expected_beam_width):
    compact_name = dataset.replace("_benchmark", "")
    results_path = (
        COMPACT_RESULTS_ROOT
        / f"{compact_name}_{run_scope}"
        / "tables"
        / "results.json"
    )
    payload = json.loads(results_path.read_text(encoding="utf-8"))
    parameters = payload.get("algorithm_parameters", payload.get("parameters", {}))
    actual_beam_width = parameters.get("heuristic", {}).get("beam_width")
    if actual_beam_width != expected_beam_width:
        raise ValueError(
            f"{dataset} {run_scope} was produced with heuristic beam width "
            f"{actual_beam_width}, expected {expected_beam_width}. Rerun 01 with B=3."
        )
    print(f"{dataset}: confirmed heuristic B={actual_beam_width}")

for dataset in COMPACT_DATASETS:
    validate_heuristic_beam_width(
        dataset,
        COMPACT_RUN_SCOPE,
        EXPECTED_HEURISTIC_BEAM_WIDTH,
    )

PROGRESS_EVERY = 100 if COMPACT_RUN_SCOPE == "full" else 25

compact_records_by_dataset = run_actual_error(
    COMPACT_DATASETS,
    comparison_results_root=COMPACT_RESULTS_ROOT,
    results_root=ACTUAL_RESULTS_ROOT,
    run_scope=COMPACT_RUN_SCOPE,
    progress_every=PROGRESS_EVERY,
)

for dataset, records in compact_records_by_dataset.items():
    print(dataset, "records:", len(records))

## Compact Surrogate Quality

This table checks whether the surrogate is numerically close to the actual physical tree error. Bias is `mean(actual - surrogate)`, so positive values mean the surrogate underestimates the actual error on average. MAE is the average absolute discrepancy. Pearson and Spearman are computed from raw per-run errors, separately by dimension and method; the pooled values combine all three methods.

In [ ]:
bias_rows = bias_mae_summary(compact_records_by_dataset)
quality_rows = surrogate_quality_summary(compact_records_by_dataset)
quality_by_method = {row["method"]: row for row in quality_rows}

print("| Dimension | Method | Bias actual-surrogate | MAE | Pearson | Spearman |")
print("|---|---|---:|---:|---:|---:|")
for row in bias_rows:
    method = row["method"]
    dim_key = row["dimension"].lower()
    quality = quality_by_method[method]
    print(
        f"| {row['dimension']} | {method} | "
        f"{row['bias']:.6f} | {row['mae']:.6f} | "
        f"{quality[f'{dim_key}_pearson']:.4f} | "
        f"{quality[f'{dim_key}_spearman']:.4f} |"
    )

all_methods = quality_by_method["All methods"]
print("\nPooled across all displayed methods")
print("| Dimension | Pearson | Spearman |")
print("|---|---:|---:|")
print(f"| 1D | {all_methods['1d_pearson']:.4f} | {all_methods['1d_spearman']:.4f} |")
print(f"| 2D | {all_methods['2d_pearson']:.4f} | {all_methods['2d_spearman']:.4f} |")

## Compact Robustness Checks

The first table recomputes Pearson and Spearman correlations within each fixed budget level, pooling methods and instances. This checks whether the high pooled correlation is merely caused by all methods improving as budget increases. The second table measures method-ranking agreement: for each instance and budget level, it asks whether the method with the lowest surrogate error is also a method with the lowest actual error.

In [ ]:
print("Correlation within fixed budget level, pooled across methods")
print("| Dimension | Budget level | Mean rho | Pearson | Spearman |")
print("|---|---:|---:|---:|---:|")
for row in within_budget_correlation_summary(compact_records_by_dataset):
    print(
        f"| {row['dimension']} | {row['budget_level']} | "
        f"{row['mean_budget_ratio']:.4f} | "
        f"{row['pearson']:.4f} | {row['spearman']:.4f} |"
    )

print("\nMethod-ranking agreement")
print("| Dimension | Budget level | Runs | Agreement rate |")
print("|---|---:|---:|---:|")
for row in method_ranking_agreement_summary(compact_records_by_dataset):
    print(
        f"| {row['dimension']} | {row['budget_level']} | "
        f"{row['runs']} | {100 * row['agreement_rate']:.1f}% |"
    )

## Compact Actual-Error Performance

This table compares algorithm conclusions after replacing the surrogate evaluation with actual Helstrom branch probabilities. The Exhaustive column is the surrogate-exhaustive solution reevaluated with actual error; it is not an actual-error optimum. Uniform and Random are included as baseline references. The Heuristic reduction column measures improvement over Greedy under actual error, and the final column measures how far Heuristic remains from the surrogate-exhaustive solution.

In [ ]:
print("| Dataset | Runs | Exhaustive actual | Greedy actual | Heuristic actual | Uniform actual | Random actual | Heuristic reduction vs Greedy | Heuristic gap vs Exhaustive |")
print("|---|---:|---:|---:|---:|---:|---:|---:|---:|")
for dataset, records in compact_records_by_dataset.items():
    row = actual_method_comparison_summary(records)
    print(
        f"| {dataset} | {row['runs']} | "
        f"{row['optimal_mean_actual_error']:.6f} | "
        f"{row['greedy_mean_actual_error']:.6f} | "
        f"{row['heuristic_mean_actual_error']:.6f} | "
        f"{row.get('uniform_mean_actual_error', float('nan')):.6f} | "
        f"{row.get('random_mean_actual_error', float('nan')):.6f} | "
        f"{row['heuristic_actual_reduction_vs_greedy']:.6f} "
        f"({row['heuristic_actual_reduction_vs_greedy_percent']:.1f}%) | "
        f"{row['heuristic_actual_gap_vs_optimal']:.6f} |"
    )

## Final Figures: Compact

This section creates the two compact-data figures used in the analysis.

**Figure 1** is the surrogate-validation scatter plot: x-axis is surrogate final tree error from the `0.5/0.5` outside-pair rule, and y-axis is actual end-to-end tree error from Helstrom branch probabilities.

**Figure 2** is the actual-error performance plot: y-axis is mean actual end-to-end error after reevaluating the saved fixed allocations and decision trees. The Exhaustive curve is the surrogate-exhaustive solution reevaluated with actual error, not an actual-error optimum.


In [ ]:
# Create, save, and display the two compact actual-error final figures inline.
for title, figure in actual_error_analysis.make_combined_actual_error_plots(
    compact_records_by_dataset,
    ACTUAL_RESULTS_ROOT,
    run_scope=COMPACT_RUN_SCOPE,
):
    display(Markdown(f"### {title}"))
    display(figure)
    plt.close(figure)


## Interpretation Summary

This section lists the figures and numerical summaries intended for the analysis and states the interpretation without overstating what the surrogate validation proves.


In [ ]:
print("Final figures to include")
print("1. Figure 1: surrogate final error vs actual final error, with 1D and 2D panels and y=x.")
print("2. Figure 2: mean actual end-to-end error vs budget ratio, with 1D and 2D panels.")

print("\nAnalysis-number checklist")
print("- Bias and MAE by dimension and method: calibration of the surrogate.")
print("- Pearson/Spearman by dimension and method, plus pooled all-method values: ranking proxy strength.")
print("- Within-budget Pearson/Spearman: checks whether pooled correlation is driven only by budget trend.")
print("- Method-ranking agreement: how often surrogate-best is also actual-best.")
print("- Mean actual errors and Heuristic reduction vs Greedy: whether conclusions survive actual evaluation.")

print("\nInterpretation patterns")
print("- High correlation supports the surrogate as a ranking proxy, not as exact calibration.")
print("- Bias > 0 means the surrogate underestimates actual error on average; bias < 0 means it overestimates.")
print("- If within-budget correlations remain high, the relationship is not only due to the shared decrease with budget.")
print("- If Heuristic actual error is below Greedy, the algorithmic ranking survives actual-error reevaluation.")
print("- The Exhaustive curve is surrogate-exhaustive reevaluated with actual error, not an actual-error optimum.")